## Silver orchestration
Reads scenes that Bronze ingested for the active AOI, crops to the AOI, converts DN to
reflectance, writes rasters to the Silver Volume and one QA row per scene to `silver.scene_qa`. 

Logic lives in `src/silver/`; this notebook only resolves config and runs.


In [0]:
%pip install rasterio tqdm

In [0]:
dbutils.library.restartPython()


In [0]:
import os
import yaml

def find_repo_root(start_dir: str, marker: str = "conf/ingestion_config.yaml", max_levels: int = 8) -> str:
    current = start_dir
    for _ in range(max_levels):
        if os.path.exists(os.path.join(current, marker)):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise FileNotFoundError(f"{marker} not found going up from {start_dir}")


notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
# notebookPath() is a logical Workspace path; open() needs the filesystem path under /Workspace
fs_path = notebook_path if notebook_path.startswith("/Workspace") else f"/Workspace{notebook_path}"
repo_root = find_repo_root(os.path.dirname(fs_path))

with open(f"{repo_root}/conf/ingestion_config.yaml") as f:
    config = yaml.safe_load(f)

print(f"active_aoi: {config['stac_discovery']['active_aoi']}")
print(f"silver bands: {config['silver']['bands']}")
